In [2]:
import os
import re
import numpy as np
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=GEMINI_API_KEY)

GEN_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "models/gemini-embedding-001"  #  embedding model name

print("Client ready.")

Client ready.


In [14]:
# Step 1: List available text files in the current directory
import glob
import os

current_dir = os.getcwd()
print(f"Current directory: {current_dir}\n")

# Find all .txt files
txt_files = glob.glob("fictional_bank_terms.txt")

if txt_files:
    print(f"📄 Found {len(txt_files)} text file(s):")
    for i, file in enumerate(txt_files, 1):
        file_size = os.path.getsize(file)
        print(f"  {i}. {file} ({file_size:,} bytes)")
    
    print(f"\n📌 Will use: {txt_files[0]} for RAG demonstration")
else:
    print("❌ No .txt files found in current directory")
    print("💡 Please add a .txt file to the current directory and run this cell again.")

Current directory: e:\Gen AI\Gen AI training_Anjishnu\Gen AI

📄 Found 1 text file(s):
  1. fictional_bank_terms.txt (1,667 bytes)

📌 Will use: fictional_bank_terms.txt for RAG demonstration


In [15]:

#Step 2: Load and chunk the selected text file
selected_file = txt_files[0]
print(f"📄 Loading file: {selected_file}\n")

# Read the file content
with open(selected_file, "r", encoding="utf-8") as f:
    file_content = f.read()

print(f"File size: {len(file_content)} characters")
print(f"Preview (first 200 chars):\n{file_content[:200]}...\n")

# Chunk the file content with larger chunks for better context
def chunk_text_file(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks"""
    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


# Create chunks from file
file_chunks = chunk_text_file(file_content, chunk_size=300, overlap=50)

# Create chunk records with metadata
file_chunk_records = []
for idx, chunk_text in enumerate(file_chunks, 1):
    file_chunk_records.append({
        "chunk_id": f"F{idx}",
        "source": selected_file,
        "text": chunk_text.strip()
    })

print(f"🧩 Created {len(file_chunk_records)} chunks from the file\n")

print("Sample chunks:")
for i, rec in enumerate(file_chunk_records[:3], 1):
    print(f"\n[{rec['chunk_id']}] ({rec['source']})")
    print(f"  Text: {rec['text'][:100]}...")

📄 Loading file: fictional_bank_terms.txt

File size: 1650 characters
Preview (first 200 chars):
TERMS OF SERVICE Fictional Bank Ltd.

Effective Date: January 1, 2026

1.  Introduction Welcome to Fictional Bank Ltd. (“Bank”, “we”, “our”,
    “us”). By accessing or using our services, you agree to...

🧩 Created 7 chunks from the file

Sample chunks:

[F1] (fictional_bank_terms.txt)
  Text: TERMS OF SERVICE Fictional Bank Ltd.

Effective Date: January 1, 2026

1.  Introduction Welcome to F...

[F2] (fictional_bank_terms.txt)
  Text: rvice.

2.  Eligibility You must be at least 18 years old and legally capable of
    entering into b...

[F3] (fictional_bank_terms.txt)
  Text: keep your account details
    updated.

4.  Services The Bank provides financial services including ...


In [16]:
# Step 3: Create embeddings for all file chunks
print("🔄 Creating embeddings for file chunks...")

def embed_texts(texts, model=EMBED_MODEL):
    """Generate embeddings for a list of texts using the Gemini embedding model."""
    result = client.models.embed_content(contents=texts, model=model)

    #Extract embeddings from the response
    embeddings = []
    if hasattr(result, 'embeddings'):
        for emb in result.embeddings:
            if hasattr(emb, 'values'):
                embeddings.append(np.array(emb.values, dtype=np.float32))
    return embeddings

file_texts = [rec["text"] for rec in file_chunk_records]
file_vectors = embed_texts(file_texts)

print(f"✅ Created {len(file_vectors)} embeddings")
print(f"📏 Embedding dimension: {len(file_vectors[0])}")
print("🚀 Ready for semantic search!") 

🔄 Creating embeddings for file chunks...
✅ Created 7 embeddings
📏 Embedding dimension: 3072
🚀 Ready for semantic search!


In [17]:
# Step 4: Create a retrieval function for the text file
def retrieve_from_file(query, top_k=3):
    """Retrieve most relevant chunks from the file based on query"""
    
    # Get query embedding
    query_vector = embed_texts([query])[0]
    
    # Calculate similarity scores
    scored_chunks = []
    for rec, vec in zip(file_chunk_records, file_vectors):
        score = cosine_similarity(query_vector, vec)
        scored_chunks.append((score, rec))
    
    # Sort by score and return top_k
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    return scored_chunks[:top_k]


# Test the retrieval with a sample query
test_query = "How to apply for a loan?"
print(f"🔍 Query: {test_query}\n")

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors (0=unrelated, 1=identical)."""
    denom=(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


results = retrieve_from_file(test_query, top_k=3)

print("📊 Top 3 relevant chunks:\n")
for rank, (score, rec) in enumerate(results, 1):
    print(f"{rank}. Score: {score:.4f} | [{rec['chunk_id']}]")
    print(f"   Text: {rec['text'][:150]}...")
    print()

🔍 Query: How to apply for a loan?

📊 Top 3 relevant chunks:

1. Score: 0.5491 | [F3]
   Text: keep your account details
    updated.

4.  Services The Bank provides financial services including savings
    accounts, loans, and digital banking t...

2. Score: 0.5287 | [F7]
   Text: at any
    time.

12. Governing Law These Terms shall be governed by the laws of the
    applicable jurisdiction.

Contact: support@fictionalbank.com...

3. Score: 0.5228 | [F5]
   Text: to:

-   Use the service for illegal purposes
-   Attempt unauthorized access to systems
-   Engage in fraudulent activities

8.  Privacy Your data wi...



In [10]:
# Step 5: Implement RAG - Retrieval-Augmented Generation
def rag_answer_from_file(query, top_k=3):
    """
    RAG pipeline:
    1. Retrieve relevant chunks from file
    2. Build context from retrieved chunks
    3. Generate answer using LLM with context
    """

    # Retrieve relevant chunks
    results = retrieve_from_file(query, top_k=top_k)

    # Build context block
    context_parts = []
    for score, rec in results:
        context_parts.append(
            f"[{rec['chunk_id']}] (relevance: {score:.2f})\n{rec['text']}"
        )

    context_block = "\n\n".join(context_parts)

    # Create prompt for LLM
    prompt = f"""You are a helpful assistant that answers questions based ONLY on the provided context.

Instructions:
- Answer the question using ONLY information from the context below
- If the context doesn’t contain enough information, say "I cannot answer based on the provided context"
- Include citations like [F1], [F2] to reference specific chunks
- Be concise and accurate

Question: {query}

Context from file:
{context_block}

Answer:"""

    # Generate answer using Gemini
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )

    answer = response.text

    # Extract citations used
    import re
    citations = sorted(set(re.findall(r"\[F\d+\]", answer)))

    return {
        "answer": answer,
        "query": query,
        "retrieved_chunks": results,
        "citations": citations
    }


print("✅ RAG function ready!")

✅ RAG function ready!


In [11]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

# List of test questions
test_questions = [
    "What are the key features of Python?",
    "How is Python used in machine learning?",
    "What are some best practices for Python programming?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)

    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)

    print(f"\n📄 Answer:")
    print(result["answer"])

    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")

    print(f"\n📊 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

🤖 RAG DEMONSTRATION - Question Answering from Text File

Question 1: What are the key features of Python?

📄 Answer:
I cannot answer based on the provided context.

🔗 Citations used: None

📊 Retrieved evidence:
  1. [F3] Relevance: 0.4893
     Preview: keep your account details
    updated.

4.  Services The Bank provides financial services including ...
  2. [F4] Relevance: 0.4850
     Preview: maintaining the
    confidentiality of your account credentials and for all activities
    under you...
  3. [F6] Relevance: 0.4743
     Preview: ot liable for indirect,
    incidental, or consequential damages arising from use of services.

10. ...

Question 2: How is Python used in machine learning?

📄 Answer:
I cannot answer based on the provided context.

🔗 Citations used: None

📊 Retrieved evidence:
  1. [F5] Relevance: 0.4659
     Preview: to:

-   Use the service for illegal purposes
-   Attempt unauthorized access to systems
-   Engage ...
  2. [F2] Relevance: 0.4627
     Preview: rvice

In [11]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

test_questions = [
    "How to apply for a loan?",
    "Any customer care details?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)

    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)

    print(f"\n📄 Answer:")
    print(result["answer"])

    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")

    print(f"\n📊 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

🤖 RAG DEMONSTRATION - Question Answering from Text File

Question 1: How to apply for a loan?

📄 Answer:
I cannot answer based on the provided context. The context mentions that the bank provides loans [F3] but does not detail the application process.

🔗 Citations used: [F3]

📊 Retrieved evidence:
  1. [F3] Relevance: 0.5491
     Preview: keep your account details
    updated.

4.  Services The Bank provides financial services including ...
  2. [F7] Relevance: 0.5287
     Preview: at any
    time.

12. Governing Law These Terms shall be governed by the laws of the
    applicable ...
  3. [F5] Relevance: 0.5228
     Preview: to:

-   Use the service for illegal purposes
-   Attempt unauthorized access to systems
-   Engage ...

Question 2: Any customer care details?

📄 Answer:
You can contact support at support@fictionalbank.com [F7].

🔗 Citations used: [F7]

📊 Retrieved evidence:
  1. [F7] Relevance: 0.5803
     Preview: at any
    time.

12. Governing Law These Terms shall be governe